# Session 15: Performance, Security Ops & DevOps Awareness

> Final hardening, CI/CD security gates, and production release readiness.

## 1. CI/CD Security Gate Pipeline

**CI/CD (Continuous Integration / Continuous Deployment)** is the practice of automatically building, testing, and deploying code on every change. A **security gate** is a mandatory step in the pipeline that blocks a deployment if a security check fails.

**Why security gates matter:**
Without automated gates, security is a manual, periodic activity — a penetration test once a year, or a code review that depends on one engineer's knowledge. With security gates in CI/CD:
- A developer gets immediate feedback if they introduce a SQL injection, a hardcoded secret, or a dependency with a known CVE.
- The cost of fixing a bug caught in CI is ~100× lower than fixing it in production.
- Security becomes part of the development workflow, not an afterthought.

**Typical security gate stages in a pipeline:**

| Stage | What it checks | Tool examples |
|---|---|---|
| Pre-commit | Secrets in staged files | `detect-secrets`, `gitleaks` |
| On PR | Static analysis (SAST) | `bandit`, `semgrep` |
| On PR | Dependency vulnerabilities | `pip-audit`, `dependabot` |
| On merge | Container image scanning | Trivy, Snyk Container |
| Pre-deploy | DAST against staging | OWASP ZAP |
| Post-deploy | Runtime monitoring | Datadog, Sentry |

**The "fail fast" principle applies to security too:** the earlier in the pipeline a check runs, the cheaper it is to fix. Pre-commit hooks are free to run; a production incident costs tens of thousands of dollars and damages user trust.

**Balancing security and velocity:** Not every finding should block a merge. Critical and High findings should be hard blockers. Medium findings can generate warnings that the team reviews in their weekly security meeting. Low findings go to the backlog. This prevents "security fatigue" where developers learn to ignore a noisy pipeline.

```yaml
# .gitlab-ci.yml — Security Gates
stages: [lint, test, security, staging, production]

sast-scan:
  stage: security
  script:
    - bandit -r ./src -f json -o sast-report.json
    - semgrep --config=auto ./src
  allow_failure: false  # BLOCKS merge if vulnerabilities found

dast-scan:
  stage: security
  script:
    - docker run owasp/zap2docker-stable zap-baseline.py -t $STAGING_URL
  only: [staging]

prod-deploy:
  stage: production
  when: manual  # Requires explicit approval
  only: [main]
  script:
    - ./scripts/deploy-blue-green.sh
```

## 2. Release Checklist (Hybrid Phase 3)

**A release checklist is a structured, repeatable process that ensures every deployment meets a minimum bar of quality, security, and operational readiness.** Without a checklist, releases depend on the memory of whoever is deploying — and people forget things, especially under pressure.

**Why "Hybrid Phase 3"?**
In a mature engineering organisation, deployments go through phases:
- **Phase 1 (manual):** Engineer deploys manually, checks a list in a document.
- **Phase 2 (automated CI/CD):** Pipeline runs tests and deploys automatically, but humans still approve.
- **Phase 3 (hybrid):** The pipeline automates everything it *can* check automatically. The checklist covers the things that require human judgment — stakeholder communication, rollback plan verification, on-call readiness.

**The categories every release checklist should cover:**

1. **Code quality gate:** All tests pass, coverage threshold met, SAST scan clean.
2. **Security gate:** No new High/Critical vulnerabilities, no secrets in code, dependencies up to date.
3. **Performance baseline:** Load test results within acceptable bounds, no regression vs. last release.
4. **Database migrations:** Backwards-compatible (old code can still run against the new schema during the deployment window), rollback script tested.
5. **Feature flags:** New features are behind a flag — can be turned off instantly without a new deployment if something goes wrong.
6. **Observability:** Dashboards updated, alert thresholds set for new metrics, runbook linked in the deployment ticket.
7. **Rollback plan:** Specific steps to revert within 15 minutes documented and tested.
8. **Communication:** Stakeholders notified, support team briefed on user-facing changes, status page updated if relevant.

**The checklist is a living document** — after every incident, add a new item that would have prevented it or detected it faster.

In [ ]:
release_checklist = {
    'Code Quality': [
        '✅ All unit tests pass (coverage >= 80%)',
        '✅ No SAST findings P0/P1',
        '✅ PR reviewed and approved',
        '✅ CHANGELOG updated',
    ],
    'Security': [
        '✅ No hardcoded secrets (git-secrets scan)',
        '✅ Dependencies vulnerability scan clean',
        '✅ DAST scan on staging passed',
        '✅ Access control reviewed',
    ],
    'Operations': [
        '✅ DB migrations tested on staging',
        '✅ Rollback procedure documented',
        '✅ Monitoring alerts configured',
        '✅ Runbook updated',
    ],
}
for category, items in release_checklist.items():
    print(f'\n== {category} ==')
    for item in items: print(f'  {item}')

## 💡 Additional Examples: Performance & DevOps

**Example 1 — Profiling before optimising**
The cardinal rule of performance work is: *measure first, optimise second*. Without profiling, developers optimise the code they think is slow — which is rarely the actual bottleneck. Python's `cProfile` and `line_profiler` show exactly which functions consume the most CPU time. For web applications, APM tools (Datadog, New Relic, Sentry Performance) show which endpoints are slow and *why* (database queries, external HTTP calls, CPU-bound processing). Optimise the largest bottleneck first — fixing a function that takes 2ms has no user-visible impact if a database query takes 2 seconds.

**Example 2 — Caching strategy: the cache hierarchy**
Caching happens at multiple levels, and each has different characteristics:
- **In-process cache (LRU dict):** Fastest (nanoseconds), but not shared between processes and lost on restart. Good for expensive computed values that are called many times per request.
- **Redis / Memcached:** Shared between all application instances, survives restarts, configurable TTL. Good for session data, rendered HTML fragments, database query results. Latency ~1ms.
- **CDN (Cloudflare, CloudFront):** Caches static assets and API responses at edge servers near the user. Latency ~5ms, no load on your servers at all.

Designing a caching strategy means deciding which data belongs at which level and what the cache invalidation rules are.

**Example 3 — Blue-green deployment for zero-downtime releases**
A blue-green deployment maintains two identical production environments ("blue" = live, "green" = new version). The new version is deployed to green and tested. Traffic is switched from blue to green at the load balancer level — typically one DNS or config change. If something goes wrong, traffic is switched back to blue in seconds. No users experience downtime, and rollback is instant. The cost is running two production environments simultaneously, which is acceptable for critical services. Feature flags achieve a similar result at the application level without duplicating infrastructure.

In [ ]:
# ─── Example 1: Caching Strategy — LRU Cache with TTL ───────────────────────
import time
import functools
from collections import OrderedDict
from typing import Any, Optional

class TTLLRUCache:
    """
    LRU Cache with Time-to-Live (TTL) expiry.
    Suitable for: API responses, DB query results, and expensive computed values.
    """
    def __init__(self, max_size: int = 128, ttl_seconds: float = 300):
        self.max_size = max_size
        self.ttl = ttl_seconds
        self._cache: OrderedDict[str, tuple[Any, float]] = OrderedDict()
        self._hits = 0
        self._misses = 0

    def get(self, key: str) -> Optional[Any]:
        if key not in self._cache:
            self._misses += 1
            return None
        value, expires_at = self._cache[key]
        if time.time() > expires_at:
            del self._cache[key]
            self._misses += 1
            return None
        # LRU: move accessed key to the end (most recently used)
        self._cache.move_to_end(key)
        self._hits += 1
        return value

    def set(self, key: str, value: Any):
        if key in self._cache:
            self._cache.move_to_end(key)
        self._cache[key] = (value, time.time() + self.ttl)
        if len(self._cache) > self.max_size:
            self._cache.popitem(last=False)  # Evict the least recently used item

    @property
    def hit_rate(self) -> float:
        total = self._hits + self._misses
        return self._hits / total if total else 0.0

    def stats(self) -> dict:
        return {'size': len(self._cache), 'hits': self._hits,
                'misses': self._misses, 'hit_rate': f'{self.hit_rate:.1%}'}

# Simulate an API layer backed by the cache
cache = TTLLRUCache(max_size=100, ttl_seconds=60)

def get_user_profile(user_id: int) -> dict:
    cache_key = f'user:{user_id}'
    cached = cache.get(cache_key)
    if cached:
        print(f'  📋 CACHE HIT  user:{user_id}')
        return cached
    # Simulate a slow DB query
    print(f'  💾 CACHE MISS user:{user_id} — querying DB...')
    profile = {'id': user_id, 'name': f'User_{user_id}', 'plan': 'premium'}
    cache.set(cache_key, profile)
    return profile

print('=== Cache Behaviour Simulation ===')
for uid in [1, 2, 1, 3, 2, 1]:   # Repeated lookups demonstrate cache hits
    get_user_profile(uid)

print(f'\nCache stats: {cache.stats()}')

# ─── Example 2: Profiling & Performance Benchmarking ─────────────────────────
import time
import cProfile
import pstats
import io
from functools import wraps

def timing(func):
    """Decorator that measures execution time — useful during development."""
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        elapsed = (time.perf_counter() - start) * 1000
        print(f'  ⏱  {func.__name__}() took {elapsed:.3f}ms')
        return result
    return wrapper

# Compare list vs set for membership lookups
@timing
def search_in_list(data: list, targets: list) -> int:
    return sum(1 for t in targets if t in data)

@timing
def search_in_set(data: set, targets: list) -> int:
    return sum(1 for t in targets if t in data)

n = 100_000
large_list = list(range(n))
large_set = set(range(n))
targets = list(range(0, n, 100))  # 1 000 lookups

print('\n=== List vs Set Lookup Benchmark ===')
list_result = search_in_list(large_list, targets)
set_result = search_in_set(large_set, targets)
assert list_result == set_result, 'Results must match!'
print(f'  Both found {list_result} matches')
print(f'  Set lookup is O(1) vs List O(n) — set wins decisively on large data\n')

# Benchmark: list comprehension vs generator for memory efficiency
@timing
def sum_with_list_comp(n: int) -> int:
    return sum([x * x for x in range(n)])  # Materialises the entire list in memory

@timing
def sum_with_generator(n: int) -> int:
    return sum(x * x for x in range(n))    # Streams values — no list allocation

print('=== List Comprehension vs Generator ===')
r1 = sum_with_list_comp(500_000)
r2 = sum_with_generator(500_000)
assert r1 == r2
print(f'  Generator uses O(1) memory vs O(n) for list comprehension')

# ─── Example 3: Blue-Green Deployment Health Check ───────────────────────────
import random
from dataclasses import dataclass
from enum import Enum

class Environment(Enum):
    BLUE  = 'blue'
    GREEN = 'green'

@dataclass
class HealthStatus:
    environment: Environment
    healthy: bool
    response_time_ms: float
    error_rate: float
    cpu_percent: float
    memory_percent: float
    active_connections: int

    def is_production_ready(self) -> bool:
        return (
            self.healthy and
            self.response_time_ms < 200 and
            self.error_rate < 0.01 and        # < 1% error threshold
            self.cpu_percent < 80 and
            self.memory_percent < 85
        )

def probe_health(env: Environment, inject_issue: bool = False) -> HealthStatus:
    """Simulate a health probe against a deployment environment."""
    base = {
        'response_time_ms': random.uniform(50, 120),
        'error_rate': random.uniform(0.001, 0.005),
        'cpu_percent': random.uniform(20, 60),
        'memory_percent': random.uniform(40, 70),
        'active_connections': random.randint(50, 200),
    }
    if inject_issue:
        base['error_rate'] = random.uniform(0.05, 0.15)     # High error rate
        base['response_time_ms'] = random.uniform(400, 800)  # Slow responses
    return HealthStatus(environment=env, healthy=not inject_issue, **base)

def blue_green_switch(current: Environment, new_env: Environment) -> dict:
    """Run health checks before switching traffic to the new environment."""
    print(f'\n🔄 Blue-Green Switch: {current.value} → {new_env.value}')
    print('   Running health checks on new environment...\n')

    # Run multiple probes for confidence
    probes = [probe_health(new_env) for _ in range(3)]
    all_healthy = all(p.is_production_ready() for p in probes)
    avg_rt = sum(p.response_time_ms for p in probes) / len(probes)
    avg_err = sum(p.error_rate for p in probes) / len(probes)

    for i, p in enumerate(probes, 1):
        status = '✅' if p.is_production_ready() else '❌'
        print(f'  Probe {i}: {status} rt={p.response_time_ms:.0f}ms, error={p.error_rate:.2%}, cpu={p.cpu_percent:.0f}%')

    if all_healthy:
        print(f'\n  ✅ All probes passed — SWITCHING traffic to {new_env.value}')
        print(f'  📊 Avg: response_time={avg_rt:.0f}ms, error_rate={avg_err:.3%}')
        return {'switched': True, 'active_env': new_env, 'reason': 'All health checks passed'}
    else:
        print(f'\n  ❌ Health check FAILED — keeping traffic on {current.value}')
        print(f'  🔔 Alert sent to on-call engineer')
        return {'switched': False, 'active_env': current, 'reason': 'New environment not production-ready'}

# Simulate a healthy deployment
print('=== Blue-Green Deployment Simulation ===')
result = blue_green_switch(current=Environment.BLUE, new_env=Environment.GREEN)
print(f'\nFinal: {result}')

# Simulate a faulty deployment that should be blocked
print('\n--- Scenario: Faulty deployment ---')
import unittest.mock as mock
with mock.patch(__name__ + '.probe_health', side_effect=lambda env, **kw: probe_health(env, inject_issue=True)):
    result2 = blue_green_switch(current=Environment.BLUE, new_env=Environment.GREEN)
    print(f'\nFinal: {result2}')


## 3. Capstone Project

### 🧪 AI Lab / Practice

> **TODO:** **GRADUATION CAPSTONE**: Build a complete module using the full Hybrid Lifecycle:

**Phase 1 (Waterfall):** Requirements doc + System design + DB schema + Security threat model

**Phase 2 (Agile):** Implement with Clean Code + Design Patterns + AI tooling. All code reviewed by CodeX/Claude before commit.

**Phase 3 (Waterfall):** Unit tests (>80% coverage) + SAST scan (zero P0/P1) + Release checklist completed

Presentation: 15-minute walkthrough covering architecture decisions, AI tools used, security measures, and lessons learned.

In [ ]:
# TODO: Implement your solution here
raise NotImplementedError('Not implemented yet — complete this exercise')